In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Resume Bullet Rewriter — From Weak to Impact-Driven

## 1. Project Overview

**Task:** Take vague, duty-focused resume bullets and rewrite them into quantified, achievement-focused statements that pass recruiter screening.

**Approach:** Prompt engineering with a local LLM — we explore zero-shot, few-shot, role-based, and rubric-guided strategies to find what produces the best rewrites.

**Stack:**
- `LangChain` + `ChatOllama` — prompt templates and LLM interaction
- `Ollama` + `qwen3.5:9b` — local LLM (no cloud API keys)

> **No API keys required.** Everything runs locally via Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Identify what makes a resume bullet **weak** vs **strong** |
| 2 | Design **zero-shot** prompts that consistently improve bullet quality |
| 3 | Use **few-shot examples** to steer style and specificity |
| 4 | Apply **role-based prompting** ("You are a career coach") for domain expertise |
| 5 | Build a **scoring rubric** and have the LLM self-evaluate its output |
| 6 | Compare **prompt strategies** side-by-side on the same inputs |
| 7 | Customize tone and vocabulary for different **industries and seniority levels** |
| 8 | Chain LLM calls: rewrite → score → refine |

## 3. Problem Statement

Most resumes fail at the bullet level. Common problems:

| Problem | Example |
|---------|---------|
| **Duty-focused** | "Responsible for managing a team" |
| **No metrics** | "Improved sales" (by how much?) |
| **Passive voice** | "Was involved in project planning" |
| **Vague verbs** | "Helped with customer issues" |
| **Too long** | 3-line run-on sentences |
| **No outcome** | "Used Python and SQL daily" (so what?) |

A strong bullet follows the **XYZ formula**: **Accomplished [X] as measured by [Y], by doing [Z].**

## 4. Why This Project Matters

- Recruiters spend **6-7 seconds** per resume — weak bullets get skipped
- ATS (Applicant Tracking Systems) filter on **action verbs** and **keywords**
- This is a high-demand use case for LLMs — LinkedIn, Teal, and Jobscan all offer AI bullet rewriters
- Prompt engineering for **constrained rewriting** (improve quality without inventing facts) is a transferable skill

## 5. Pipeline Architecture

```
Weak Resume Bullet
      │
      ▼
  [Diagnose] ── identify what's wrong (vague verb, no metric, passive)
      │
      ▼
  [Rewrite] ── transform into impact-driven format
      │
      ├──▶ Strategy A: Zero-shot (simple instruction)
      ├──▶ Strategy B: Few-shot (with good/bad examples)
      ├──▶ Strategy C: Role-based ("You are a senior recruiter")
      └──▶ Strategy D: Rubric-guided (score-then-rewrite)
      │
      ▼
  [Score] ── evaluate rewrite quality (1-10 rubric)
      │
      ▼
  [Before / After] ── side-by-side comparison
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core

print("Dependencies: langchain, langchain-ollama")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

In [ ]:
import os
import json
import re
import warnings
import time
from textwrap import dedent

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

print("All imports OK")

## 8. Configuration & Helpers

In [ ]:
LLM_MODEL = "qwen3.5:9b"

llm = ChatOllama(model=LLM_MODEL, temperature=0)
llm_creative = ChatOllama(model=LLM_MODEL, temperature=0.4)


def clean(text: str) -> str:
    """Strip thinking tags from model output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str) -> dict | None:
    """Extract JSON from LLM output."""
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    end = text.rfind("}") + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(system: str, user: str, creative: bool = False) -> str:
    """Send a prompt and return cleaned text."""
    model = llm_creative if creative else llm
    resp = model.invoke([SystemMessage(content=system), HumanMessage(content=user)])
    return clean(resp.content)


# Quick connectivity test
test = ask("Reply with one word.", "Say ready.")
print(f"LLM ready: {test[:30]}")

## 9. Input Dataset — Weak Resume Bullets

Each bullet includes the role context and the original weak text. Ground-truth issues are labeled so we can evaluate whether rewrites fix the right problems.

In [ ]:
WEAK_BULLETS = [
    # ── Software Engineering ──────────────────────────
    {
        "id": 1,
        "role": "Software Engineer",
        "bullet": "Responsible for developing backend services",
        "issues": ["duty-focused", "no metrics", "vague verb"],
    },
    {
        "id": 2,
        "role": "Software Engineer",
        "bullet": "Worked on improving the performance of the application",
        "issues": ["vague verb", "no metrics", "no outcome"],
    },
    {
        "id": 3,
        "role": "Full Stack Developer",
        "bullet": "Used React and Node.js to build features for the website",
        "issues": ["no outcome", "no metrics", "listing tools not impact"],
    },

    # ── Data / ML ─────────────────────────────────────
    {
        "id": 4,
        "role": "Data Analyst",
        "bullet": "Created reports and dashboards for the management team",
        "issues": ["no metrics", "no outcome", "vague scope"],
    },
    {
        "id": 5,
        "role": "Machine Learning Engineer",
        "bullet": "Built machine learning models to predict things",
        "issues": ["extremely vague", "no metrics", "no specifics"],
    },

    # ── Product / Management ──────────────────────────
    {
        "id": 6,
        "role": "Product Manager",
        "bullet": "Managed the product roadmap and coordinated with engineering",
        "issues": ["duty-focused", "no outcome", "passive"],
    },
    {
        "id": 7,
        "role": "Project Manager",
        "bullet": "Was involved in planning and executing projects on time",
        "issues": ["passive voice", "vague verb", "no specifics"],
    },

    # ── Marketing / Sales ─────────────────────────────
    {
        "id": 8,
        "role": "Marketing Manager",
        "bullet": "Helped with social media campaigns and content creation",
        "issues": ["vague verb", "no metrics", "no ownership"],
    },
    {
        "id": 9,
        "role": "Sales Representative",
        "bullet": "Responsible for reaching out to potential customers and closing deals",
        "issues": ["duty-focused", "no metrics", "no outcome"],
    },

    # ── Operations / Support ──────────────────────────
    {
        "id": 10,
        "role": "Customer Support Specialist",
        "bullet": "Answered customer emails and phone calls",
        "issues": ["duty-focused", "no metrics", "no impact"],
    },
    {
        "id": 11,
        "role": "Operations Analyst",
        "bullet": "Assisted in streamlining internal processes using Excel and Python",
        "issues": ["vague verb", "no metrics", "listing tools"],
    },
    {
        "id": 12,
        "role": "HR Coordinator",
        "bullet": "Handled recruitment and onboarding of new employees",
        "issues": ["duty-focused", "no metrics", "generic"],
    },
]

print(f"Dataset: {len(WEAK_BULLETS)} weak bullets across {len(set(b['role'] for b in WEAK_BULLETS))} roles")
for b in WEAK_BULLETS:
    print(f"  #{b['id']:2d} [{b['role']:30s}] {b['bullet']}")

## 10. What Makes a Strong Bullet?

Before coding, let's define the target quality. A **strong** resume bullet has:

| Element | Description | Example |
|---------|-------------|---------|
| **Action verb** | Starts with a powerful, specific verb | *Architected*, *Spearheaded*, *Reduced* |
| **Quantified impact** | Numbers, percentages, dollar amounts, time saved | *by 40%*, *saving $120K/year*, *across 15 markets* |
| **Scope / context** | Team size, user base, system scale | *for 2M+ users*, *leading a team of 8* |
| **Outcome** | What changed because of this work? | *reducing churn by 18%*, *accelerating deployment by 3x* |
| **Conciseness** | One bullet = one achievement, ≤ 2 lines | No run-on sentences |

**The XYZ Formula:** Accomplished **[X]** as measured by **[Y]**, by doing **[Z]**.

---

# Part A — Prompt Strategies

We test four different prompting strategies on the same inputs and compare results.

## 11. Strategy 1: Zero-Shot Rewrite

The simplest approach — just tell the LLM what to do. No examples, no persona.

In [ ]:
ZERO_SHOT_SYSTEM = """Rewrite weak resume bullet points into strong, impact-driven statements.

Rules:
- Start with a strong action verb (never "Responsible for" or "Helped with")
- Include a quantified metric or plausible placeholder like [X%] if the original has none
- Show the outcome or business impact
- Keep it to one concise sentence (max 25 words)
- Do NOT invent specific company names, product names, or technologies not in the original
- Return ONLY the rewritten bullet, nothing else"""

ZERO_SHOT_USER = """Role: {role}
Original bullet: {bullet}

Rewritten bullet:"""

print("STRATEGY 1: ZERO-SHOT")
print("=" * 70)

zero_shot_results = []
for b in WEAK_BULLETS:
    rewrite = ask(ZERO_SHOT_SYSTEM, ZERO_SHOT_USER.format(role=b["role"], bullet=b["bullet"]))
    zero_shot_results.append({"id": b["id"], "original": b["bullet"], "rewrite": rewrite})
    print(f"#{b['id']:2d} BEFORE: {b['bullet']}")
    print(f"    AFTER:  {rewrite}")
    print()

## 12. Strategy 2: Few-Shot Rewrite

We give the LLM concrete before/after examples so it can mimic the style and specificity level. Few-shot examples are the single most effective way to steer LLM output quality.

**Key design decision:** The examples should cover different *types* of weakness (vague verb, no metric, passive voice) so the LLM generalizes.

In [ ]:
FEW_SHOT_SYSTEM = """Rewrite weak resume bullets into strong, impact-driven statements.
- Start with a strong action verb
- Add metrics or plausible placeholders [X%]
- Show outcome / business impact
- Max 25 words, one sentence
- Return ONLY the rewritten bullet"""

FEW_SHOT_USER = """Here are examples of good rewrites:

WEAK: "Responsible for managing a team of developers"
STRONG: "Led a team of 8 engineers delivering 12 features per quarter, reducing release cycle time by 30%"

WEAK: "Worked on improving website performance"
STRONG: "Optimized page load speed by 55% (3.2s → 1.4s), increasing conversion rate by 12% across 500K monthly visitors"

WEAK: "Helped with data analysis"
STRONG: "Built automated analytics pipeline processing 2M+ daily records, cutting reporting turnaround from 5 days to 4 hours"

Now rewrite this bullet:

Role: {role}
WEAK: "{bullet}"
STRONG:"""

print("STRATEGY 2: FEW-SHOT")
print("=" * 70)

few_shot_results = []
for b in WEAK_BULLETS:
    rewrite = ask(FEW_SHOT_SYSTEM, FEW_SHOT_USER.format(role=b["role"], bullet=b["bullet"]))
    # Strip quotes if LLM wraps in them
    rewrite = rewrite.strip('"').strip("'")
    few_shot_results.append({"id": b["id"], "original": b["bullet"], "rewrite": rewrite})
    print(f"#{b['id']:2d} BEFORE: {b['bullet']}")
    print(f"    AFTER:  {rewrite}")
    print()

## 13. Strategy 3: Role-Based Prompting

Assign the LLM a **persona** — a senior recruiter or career coach. The persona activates domain-specific knowledge about what hiring managers look for.

In [ ]:
ROLE_SYSTEM = """You are a senior technical recruiter at a FAANG company with 15 years of 
experience screening resumes. You've reviewed over 50,000 resumes and know exactly what 
makes a bullet point stand out vs get skipped.

When you rewrite a bullet:
- You replace vague verbs with power verbs (Architected, Spearheaded, Drove, Reduced)
- You demand metrics — if none exist, add realistic placeholders [X%]
- You always show business impact (revenue, efficiency, user growth, cost savings)
- You keep it punchy: one sentence, max 25 words
- You never start with "Responsible for" — that's the #1 red flag you tell candidates about

Return ONLY the rewritten bullet point, nothing else."""

ROLE_USER = """A candidate for {role} submitted this bullet:
"{bullet}"

Rewrite it so it passes your 6-second screening test:"""

print("STRATEGY 3: ROLE-BASED (Senior Recruiter)")
print("=" * 70)

role_results = []
for b in WEAK_BULLETS:
    rewrite = ask(ROLE_SYSTEM, ROLE_USER.format(role=b["role"], bullet=b["bullet"]))
    rewrite = rewrite.strip('"').strip("'")
    role_results.append({"id": b["id"], "original": b["bullet"], "rewrite": rewrite})
    print(f"#{b['id']:2d} BEFORE: {b['bullet']}")
    print(f"    AFTER:  {rewrite}")
    print()

## 14. Strategy 4: Rubric-Guided (Score → Diagnose → Rewrite)

The most sophisticated strategy: first score the original bullet, diagnose its weaknesses, then rewrite with specific improvements. This chain-of-thought approach produces the most targeted rewrites.

In [ ]:
RUBRIC_SYSTEM = """You are a resume quality evaluator and rewriter.

Score each bullet on these 5 criteria (1-10 each):
1. ACTION VERB — Does it start with a strong, specific verb? (not "Responsible", "Helped", "Worked on")
2. METRICS — Does it include numbers, percentages, or dollar amounts?
3. IMPACT — Does it show a business outcome (not just a task)?
4. SPECIFICITY — Is it concrete and unique (vs generic and copy-pasteable)?
5. CONCISENESS — Is it ≤25 words, one clear sentence?

Return ONLY valid JSON, no other text."""

RUBRIC_USER = """Evaluate and rewrite this resume bullet.

Role: {role}
Bullet: "{bullet}"

Return JSON:
{{
  "scores": {{
    "action_verb": <1-10>,
    "metrics": <1-10>,
    "impact": <1-10>,
    "specificity": <1-10>,
    "conciseness": <1-10>
  }},
  "total_score": <sum out of 50>,
  "weaknesses": ["list of specific issues"],
  "rewrite": "the improved bullet point"
}}"""

print("STRATEGY 4: RUBRIC-GUIDED")
print("=" * 70)

rubric_results = []
for b in WEAK_BULLETS:
    resp = ask(RUBRIC_SYSTEM, RUBRIC_USER.format(role=b["role"], bullet=b["bullet"]))
    parsed = parse_json(resp)
    if parsed:
        scores = parsed.get("scores", {})
        total = parsed.get("total_score", sum(scores.values()) if scores else 0)
        rewrite = parsed.get("rewrite", "(no rewrite)")
        weaknesses = parsed.get("weaknesses", [])
    else:
        total, rewrite, weaknesses, scores = 0, resp, [], {}

    rubric_results.append({
        "id": b["id"], "original": b["bullet"], "rewrite": rewrite,
        "scores": scores, "total": total, "weaknesses": weaknesses,
    })

    print(f"#{b['id']:2d} [{b['role']}]")
    print(f"    BEFORE:     {b['bullet']}")
    print(f"    SCORE:      {total}/50  {scores}")
    print(f"    WEAKNESSES: {', '.join(weaknesses[:3])}")
    print(f"    AFTER:      {rewrite}")
    print()

---

# Part B — Comparison & Evaluation

## 15. Before / After Comparison — All Strategies Side by Side

In [ ]:
print("BEFORE / AFTER — ALL STRATEGIES")
print("=" * 80)

for i, b in enumerate(WEAK_BULLETS):
    print(f"\n{'─' * 80}")
    print(f"#{b['id']} | {b['role']} | Issues: {', '.join(b['issues'])}")
    print(f"{'─' * 80}")
    print(f"  ORIGINAL:    {b['bullet']}")
    print(f"  ZERO-SHOT:   {zero_shot_results[i]['rewrite']}")
    print(f"  FEW-SHOT:    {few_shot_results[i]['rewrite']}")
    print(f"  ROLE-BASED:  {role_results[i]['rewrite']}")
    print(f"  RUBRIC:      {rubric_results[i]['rewrite']}")

## 16. Scoring Rubric — Rate the Rewrites

We now use the LLM as a judge to score every rewrite from every strategy. This lets us quantitatively compare strategies.

In [ ]:
JUDGE_SYSTEM = """You are a resume quality judge. Score the given bullet point on a scale of 1-10.

Scoring criteria:
- 1-3: Weak (vague, no metrics, passive, duty-focused)
- 4-6: Mediocre (some improvement but still missing metrics or impact)
- 7-8: Strong (action verb, metrics, clear impact)
- 9-10: Exceptional (compelling, quantified, specific, concise)

Return ONLY a JSON object: {"score": <1-10>, "reason": "one sentence"}"""

JUDGE_USER = """Role: {role}
Bullet: "{bullet}"

Score this bullet:"""


def score_bullet(role: str, bullet: str) -> dict:
    """Score a single bullet using LLM-as-judge."""
    resp = ask(JUDGE_SYSTEM, JUDGE_USER.format(role=role, bullet=bullet))
    parsed = parse_json(resp)
    if parsed and "score" in parsed:
        return parsed
    return {"score": 0, "reason": "parse error"}


# Score originals and all rewrites
print("Scoring all bullets (originals + 4 strategies × 12 bullets = 60 calls)...\n")

strategy_names = ["original", "zero_shot", "few_shot", "role_based", "rubric"]
all_strategy_results = [
    [{"rewrite": b["bullet"]} for b in WEAK_BULLETS],  # originals
    zero_shot_results,
    few_shot_results,
    role_results,
    rubric_results,
]

score_table = {name: [] for name in strategy_names}

for s_idx, (s_name, s_results) in enumerate(zip(strategy_names, all_strategy_results)):
    for b_idx, b in enumerate(WEAK_BULLETS):
        bullet_text = s_results[b_idx]["rewrite"] if s_name != "original" else b["bullet"]
        result = score_bullet(b["role"], bullet_text)
        score_table[s_name].append(result["score"])
    avg = sum(score_table[s_name]) / len(score_table[s_name])
    print(f"  {s_name:12s}: avg score = {avg:.1f}/10")

print("\nScoring complete!")

In [ ]:
# ── Detailed score table ──────────────────────────────
print(f"\n{'Bullet':>8s} | {'Original':>8s} | {'ZeroShot':>8s} | {'FewShot':>8s} | {'RoleBsd':>8s} | {'Rubric':>8s} | {'Best Strategy':>14s}")
print("-" * 85)

strategy_wins = {name: 0 for name in strategy_names[1:]}  # skip original

for i, b in enumerate(WEAK_BULLETS):
    scores = {name: score_table[name][i] for name in strategy_names}
    rewrite_scores = {name: score_table[name][i] for name in strategy_names[1:]}
    best = max(rewrite_scores, key=rewrite_scores.get)
    strategy_wins[best] += 1

    row = " | ".join(f"{scores[n]:>8d}" for n in strategy_names)
    print(f"  #{b['id']:2d}    | {row} | {best:>14s}")

print("-" * 85)
avgs = " | ".join(f"{sum(score_table[n])/len(score_table[n]):>8.1f}" for n in strategy_names)
print(f"  {'AVG':>6s} | {avgs} |")

print(f"\nStrategy wins (best score on most bullets):")
for name, wins in sorted(strategy_wins.items(), key=lambda x: -x[1]):
    print(f"  {name:12s}: {wins} wins")

## 17. Score Improvement Analysis

How much does each strategy improve over the original?

In [ ]:
print("SCORE IMPROVEMENT OVER ORIGINAL")
print("=" * 50)

for name in strategy_names[1:]:
    improvements = [
        score_table[name][i] - score_table["original"][i]
        for i in range(len(WEAK_BULLETS))
    ]
    avg_imp = sum(improvements) / len(improvements)
    max_imp = max(improvements)
    min_imp = min(improvements)
    declined = sum(1 for x in improvements if x < 0)

    print(f"\n{name}:")
    print(f"  Avg improvement: +{avg_imp:.1f} points")
    print(f"  Best improvement: +{max_imp}")
    print(f"  Worst:            {'+' if min_imp >= 0 else ''}{min_imp}")
    print(f"  Declined (worse than original): {declined}/{len(WEAK_BULLETS)}")

---

# Part C — Advanced Features

## 18. Rewrite → Score → Refine Loop

If the first rewrite scores below 8, ask the LLM to refine it further. This iterative approach catches issues the initial rewrite missed.

In [ ]:
REFINE_USER = """This resume bullet scored {score}/10. The feedback was: "{reason}"

Current bullet: "{bullet}"
Role: {role}

Improve it further. Fix the specific issues mentioned. Return ONLY the improved bullet."""


def rewrite_with_refinement(role: str, bullet: str, threshold: int = 8, max_rounds: int = 3) -> dict:
    """Rewrite, score, and refine until quality threshold is met."""
    # Initial rewrite
    current = ask(ROLE_SYSTEM, ROLE_USER.format(role=role, bullet=bullet)).strip('"')
    history = [{"round": 0, "bullet": bullet, "score": None, "type": "original"}]

    for round_num in range(1, max_rounds + 1):
        result = score_bullet(role, current)
        score = result["score"]
        reason = result.get("reason", "")
        history.append({"round": round_num, "bullet": current, "score": score, "type": "rewrite"})

        if score >= threshold:
            break

        # Refine
        current = ask(
            ROLE_SYSTEM,
            REFINE_USER.format(score=score, reason=reason, bullet=current, role=role),
        ).strip('"')

    return {"final": current, "history": history}


# Demo: refine the weakest bullets
print("REWRITE → SCORE → REFINE LOOP")
print("=" * 70)

for b in WEAK_BULLETS[:4]:
    print(f"\n#{b['id']} [{b['role']}]")
    print(f"  Original: {b['bullet']}")
    result = rewrite_with_refinement(b["role"], b["bullet"])
    for h in result["history"]:
        if h["score"] is not None:
            print(f"  Round {h['round']}: (score {h['score']}/10) {h['bullet']}")
    print(f"  ✅ Final: {result['final']}")

## 19. Industry-Specific Rewriting

Different industries value different things. A tech resume emphasizes scalability and performance; a finance resume emphasizes compliance and risk reduction.

In [ ]:
INDUSTRY_PROFILES = {
    "tech": {
        "emphasis": "scalability, system performance, user growth, deployment speed, technical complexity",
        "verbs": "Architected, Engineered, Optimized, Deployed, Scaled, Automated",
        "metrics": "latency (ms), throughput (req/s), uptime (%), user count, deploy frequency",
    },
    "finance": {
        "emphasis": "risk reduction, regulatory compliance, portfolio performance, cost savings, accuracy",
        "verbs": "Mitigated, Audited, Optimized, Forecasted, Streamlined, Governed",
        "metrics": "AUM ($), risk reduction (%), compliance rate, cost savings ($), accuracy (%)",
    },
    "healthcare": {
        "emphasis": "patient outcomes, compliance, safety, efficiency, quality of care",
        "verbs": "Improved, Implemented, Ensured, Reduced, Coordinated, Standardized",
        "metrics": "patient satisfaction (%), readmission rate, wait time, compliance rate",
    },
}

INDUSTRY_USER = """Rewrite this resume bullet for a {industry} industry role.

Industry context:
- Emphasized qualities: {emphasis}
- Preferred verbs: {verbs}
- Typical metrics: {metrics}

Role: {role}
Original: "{bullet}"

Rewrite for {industry} (one sentence, max 25 words, ONLY the bullet):"""

# Take a generic bullet and rewrite for each industry
generic = WEAK_BULLETS[3]  # Data Analyst: "Created reports and dashboards"
print(f"ORIGINAL: {generic['bullet']}")
print(f"ROLE:     {generic['role']}")
print()

for industry, profile in INDUSTRY_PROFILES.items():
    rewrite = ask(
        FEW_SHOT_SYSTEM,
        INDUSTRY_USER.format(
            industry=industry,
            role=generic["role"],
            bullet=generic["bullet"],
            **profile,
        ),
    ).strip('"')
    print(f"  {industry.upper():12s}: {rewrite}")

## 20. Seniority-Adaptive Rewriting

Junior bullets emphasize learning and contribution; senior bullets emphasize leadership and strategic impact.

In [ ]:
SENIORITY_PROFILES = {
    "junior": "Emphasize learning, contribution, and ownership of tasks. Focus on skills applied and growth.",
    "mid-level": "Emphasize ownership of projects, cross-team collaboration, and measurable results.",
    "senior": "Emphasize strategic impact, leadership, system-level decisions, mentorship, and organizational outcomes.",
}

SENIORITY_USER = """Rewrite this bullet for a {level} {role}.

Seniority guidance: {guidance}

Original: "{bullet}"

Rewrite (one sentence, max 25 words, ONLY the bullet):"""

test_bullet = WEAK_BULLETS[0]  # SWE: "Responsible for developing backend services"
print(f"ORIGINAL: {test_bullet['bullet']}")
print(f"ROLE:     {test_bullet['role']}")
print()

for level, guidance in SENIORITY_PROFILES.items():
    rewrite = ask(
        ROLE_SYSTEM,
        SENIORITY_USER.format(
            level=level,
            role=test_bullet["role"],
            guidance=guidance,
            bullet=test_bullet["bullet"],
        ),
    ).strip('"')
    print(f"  {level.upper():12s}: {rewrite}")

print("\nNotice: junior = skills + learning, mid = ownership + metrics, senior = strategy + org impact")

## 21. Batch Processor — Rewrite an Entire Resume

A practical function that takes a list of bullets (as a user might paste them) and returns a fully polished set.

In [ ]:
def rewrite_resume(role: str, bullets: list[str], strategy: str = "few_shot") -> list[dict]:
    """
    Rewrite a list of resume bullets using the specified strategy.
    Returns list of {original, rewrite, score} dicts.
    """
    system = FEW_SHOT_SYSTEM if strategy == "few_shot" else ROLE_SYSTEM
    user_tmpl = FEW_SHOT_USER if strategy == "few_shot" else ROLE_USER

    results = []
    for bullet in bullets:
        rewrite = ask(system, user_tmpl.format(role=role, bullet=bullet)).strip('"')
        score = score_bullet(role, rewrite)
        results.append({
            "original": bullet,
            "rewrite": rewrite,
            "score": score["score"],
        })
    return results


# ── Demo: rewrite a full experience section ───────────
sample_resume = [
    "Responsible for writing Python code for data pipelines",
    "Worked with the ML team on model deployment",
    "Created documentation for internal tools",
    "Participated in code reviews and sprint planning",
    "Maintained databases and performed data migrations",
]

print("BATCH RESUME REWRITE")
print(f"Role: Data Engineer")
print("=" * 70)

polished = rewrite_resume("Data Engineer", sample_resume)
for p in polished:
    print(f"\n  BEFORE: {p['original']}")
    print(f"  AFTER:  {p['rewrite']}")
    print(f"  SCORE:  {p['score']}/10")

avg_score = sum(p["score"] for p in polished) / len(polished)
print(f"\n  Average quality: {avg_score:.1f}/10")

## 22. Error Analysis

What goes wrong and why?

In [ ]:
# ── Edge case 1: Already strong bullet ────────────────
print("EDGE CASE 1: What happens when the bullet is already strong?")
print("-" * 60)
strong_bullet = "Reduced API latency by 62% (450ms → 170ms) by implementing Redis caching, serving 2M+ daily requests"
rewrite = ask(ROLE_SYSTEM, ROLE_USER.format(role="Backend Engineer", bullet=strong_bullet)).strip('"')
print(f"  ORIGINAL:  {strong_bullet}")
print(f"  REWRITTEN: {rewrite}")
print(f"  → Ideally the LLM should make minimal changes or say it's already strong.")

# ── Edge case 2: Extremely vague single-word bullet ───
print(f"\nEDGE CASE 2: Extremely vague bullet")
print("-" * 60)
vague = "Did stuff"
rewrite2 = ask(ROLE_SYSTEM, ROLE_USER.format(role="Software Engineer", bullet=vague)).strip('"')
print(f"  ORIGINAL:  {vague}")
print(f"  REWRITTEN: {rewrite2}")
print(f"  → Too vague to rewrite meaningfully. LLM has to hallucinate everything.")

# ── Edge case 3: Non-English bullet ───────────────────
print(f"\nEDGE CASE 3: Non-English bullet")
print("-" * 60)
french = "Responsable de la gestion des bases de données"
rewrite3 = ask(ROLE_SYSTEM, ROLE_USER.format(role="DBA", bullet=french)).strip('"')
print(f"  ORIGINAL:  {french}")
print(f"  REWRITTEN: {rewrite3}")
print(f"  → Model may translate + rewrite, or just rewrite in French.")

## 23. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **No examples in prompt** | LLM guesses what "strong" means differently each time | Give 2-3 before/after examples (few-shot) |
| **Accepting hallucinated metrics** | LLM invents "saved $2.4M" when the user did an internship | Use `[X%]` placeholders the user fills in |
| **One prompt for everything** | Rewrite + score + diagnose in one call degrades quality | Chain: rewrite → score → refine |
| **Ignoring role context** | A PM bullet sounds wrong on an engineer resume | Always include the target role in the prompt |
| **No length constraint** | LLM produces 3-sentence paragraphs | Specify "max 25 words" or "one sentence" |
| **Using temperature=1** | Every run produces different rewrites of varying quality | Use temperature 0-0.4 for consistency |
| **Not testing on strong bullets** | If you pass a great bullet, it should stay mostly unchanged | Add guard: score first, skip if ≥8 |
| **Ignoring industry context** | "Spearheaded" is great for tech, odd for healthcare | Customize verbs and metrics per industry |

## 24. Production Improvement Ideas

1. **Interactive web app** — Paste resume bullets, get instant rewrites (Streamlit/Gradio)
2. **ATS keyword matching** — Compare rewrite against job description for keyword coverage
3. **Template library** — Pre-built XYZ templates per role (SWE, PM, Marketing, etc.)
4. **User feedback loop** — Let users rate rewrites; use top-rated ones as few-shot examples
5. **Fine-tuned model** — Train on thousands of weak→strong pairs from career coaching datasets
6. **Resume-wide coherence** — Ensure bullets across sections tell a consistent career story
7. **Job description alignment** — Tailor rewrites to match specific job posting language
8. **Multi-language support** — Detect language, translate, rewrite in English or original language

## 25. Exercises — Try These Yourself

### Exercise 1: Your Own Bullets

Replace the bullets below with real ones from your resume and run them through the pipeline.

In [ ]:
# ── Exercise 1: Rewrite YOUR resume ───────────────────
MY_ROLE = "Data Scientist"  # ← Change to your role
MY_BULLETS = [
    "Worked on machine learning models for the team",     # ← Replace with yours
    "Did data analysis and created visualizations",       # ← Replace with yours
    "Presented findings to stakeholders",                 # ← Replace with yours
]

print(f"Your role: {MY_ROLE}")
print("=" * 70)
results = rewrite_resume(MY_ROLE, MY_BULLETS)
for r in results:
    print(f"\n  BEFORE: {r['original']}")
    print(f"  AFTER:  {r['rewrite']}")
    print(f"  SCORE:  {r['score']}/10")

### Exercise 2: Custom Tone — Executive vs Startup vs Academic

In [ ]:
# ── Exercise 2: Tone customization ────────────────────
TONE_PROFILES = {
    "executive": "Use strategic language: 'Drove', 'Transformed', 'Championed'. Focus on org-wide impact, revenue, and team scale.",
    "startup": "Use scrappy, high-energy language: 'Shipped', 'Hacked', 'Bootstrapped'. Focus on speed, wearing multiple hats, and 0-to-1 building.",
    "academic": "Use methodical language: 'Investigated', 'Published', 'Developed methodology'. Focus on research contributions and novel findings.",
}

TONE_USER = """Rewrite this bullet in a {tone} tone.
Tone guidance: {guidance}

Role: {role}
Original: "{bullet}"

Rewrite (one sentence, max 25 words, ONLY the bullet):"""

test = WEAK_BULLETS[1]  # "Worked on improving the performance of the application"
print(f"ORIGINAL: {test['bullet']}")
print(f"ROLE:     {test['role']}")
print()

for tone, guidance in TONE_PROFILES.items():
    rewrite = ask(
        ROLE_SYSTEM,
        TONE_USER.format(tone=tone, guidance=guidance, role=test["role"], bullet=test["bullet"]),
    ).strip('"')
    print(f"  {tone.upper():12s}: {rewrite}")

print("\nTry creating your own tone: 'government', 'nonprofit', or 'creative agency'!")

### Exercise 3: Build a Self-Improving Rewriter

Chain three LLM calls: Diagnose → Rewrite → Critic. Have the critic provide actionable feedback.

In [ ]:
# ── Exercise 3: 3-step chain ──────────────────────────

# Step 1: Diagnose
test = WEAK_BULLETS[5]  # PM: "Managed the product roadmap"
diagnose = ask(
    "You are a resume critic. List the specific weaknesses of this bullet in 2-3 short points.",
    f'Role: {test["role"]}\nBullet: "{test["bullet"]}"',
)
print("STEP 1 — DIAGNOSE:")
print(f"  Bullet: {test['bullet']}")
print(f"  Diagnosis: {diagnose}")

# Step 2: Rewrite (informed by diagnosis)
rewrite = ask(
    ROLE_SYSTEM,
    f'Fix these specific issues: {diagnose}\n\nOriginal: "{test["bullet"]}"\nRole: {test["role"]}\n\nRewrite (one sentence, ONLY the bullet):',
).strip('"')
print(f"\nSTEP 2 — REWRITE:")
print(f"  {rewrite}")

# Step 3: Critic feedback
critic = ask(
    "You are a harsh but fair resume critic. Score 1-10 and give one sentence of remaining feedback.",
    f'Role: {test["role"]}\nBullet: "{rewrite}"\nScore and critique:',
)
print(f"\nSTEP 3 — CRITIC:")
print(f"  {critic}")
print("\nTry extending this to a loop: keep refining until the critic gives 9+/10!")

### Mini Challenge

1. **Role-swap test:** Take 3 bullets from your dataset and rewrite the same bullet for Software Engineer, Product Manager, and Marketing Manager. How does the same achievement get framed differently?

2. **Prompt ablation:** Remove one element at a time from the few-shot prompt (remove examples, remove length constraint, remove the instruction to include metrics). How much does quality drop?

3. **Build a scorer calibration:** Take 10 bullets you consider strong (score 9-10) and 10 weak ones (score 2-4). Run the scoring rubric on all 20. Does the LLM judge agree with your human judgment?

4. **ATS keyword integration:** Given a job description, add an extra rewrite step that injects relevant keywords from the JD into the bullet without making it sound unnatural.

## 26. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)

print(f"\nDataset: {len(WEAK_BULLETS)} weak bullets across {len(set(b['role'] for b in WEAK_BULLETS))} roles")
print(f"Strategies tested: 4 (zero-shot, few-shot, role-based, rubric-guided)")

print(f"\nAverage scores by strategy:")
for name in strategy_names:
    avg = sum(score_table[name]) / len(score_table[name])
    label = "(baseline)" if name == "original" else ""
    print(f"  {name:12s}: {avg:.1f}/10 {label}")

best_strategy = max(strategy_names[1:], key=lambda n: sum(score_table[n]))
print(f"\nBest performing strategy: {best_strategy}")

print(f"\nAdditional features demonstrated:")
print(f"  - Iterative refinement loop (rewrite → score → refine)")
print(f"  - Industry-specific rewriting (tech / finance / healthcare)")
print(f"  - Seniority-adaptive rewriting (junior / mid / senior)")
print(f"  - Batch resume processor")
print(f"\nLLM: {LLM_MODEL}")

## 27. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Few-shot examples** are the single biggest quality lever — 2-3 good before/after pairs steer style and specificity |
| 2 | **Role-based prompting** ("You are a senior recruiter") activates domain expertise the LLM already has |
| 3 | **The XYZ formula** (Accomplished X measured by Y, by doing Z) is a reliable target format |
| 4 | **Scoring rubrics** make quality measurable and enable automated refinement loops |
| 5 | **Temperature matters**: 0 for classification/scoring, 0.3-0.5 for generation |
| 6 | **Chaining** (diagnose → rewrite → critique) produces better results than a single call |
| 7 | **Industry and seniority context** dramatically change what makes a bullet strong |
| 8 | **Never trust hallucinated metrics** — LLMs will invent numbers; use placeholders the user fills in |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*